# Tutorial - Assembling a Rotor


This tutorial shows how to assemble shaft, disk, bearing and point-mass elements into a `Rotor` and more.

`Rotor` is the main class from ROSS. It takes as argument lists with all elements and assembles the mass, gyroscopic, damping and stiffness global matrices for the system. The object created has several methods that can be used to evaluate the dynamics of the model (they all start with the prefix `.run_`).

To use this class, you must input all the already instantiated elements in a list format.

If the shaft elements are not numbered, the class set a number for each one, according to the element's position in the list supplied to the rotor constructor.

To assemble the matrices, the `Rotor` class takes the local DoF's index from each element (element method `.dof_mapping()`) and calculate the global index.

For two or more shafts coupled by gears, see [Tutorial - MultiRotor System](tutorial_multirotor.ipynb).


In [1]:
import numpy as np
import ross as rs

# Make sure the default renderer is set to 'notebook_connected' for inline plots in Jupyter
import plotly.io as pio

pio.renderers.default = "notebook_connected"

steel = rs.Material(name="steel", rho=7810, E=211e9, G_s=81.2e9)

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Section 1: Creating a rotor model

Let's create a simple rotor model with $1.5$ m length with 6 identical shaft elements, 2 disks, 2 bearings in the shaft ends and a support linked to the first bearing. First, we create the elements, then we input them to the `Rotor` class.

In [2]:
n = 6

shaft_elem = [
    rs.ShaftElement(
        L=0.25,
        idl=0.0,
        odl=0.05,
        material=steel,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for _ in range(n)
]

disk0 = rs.DiskElement.from_geometry(
    n=2, material=steel, width=0.07, i_d=0.05, o_d=0.28
)
disk1 = rs.DiskElement.from_geometry(
    n=4, material=steel, width=0.07, i_d=0.05, o_d=0.28
)
disks = [disk0, disk1]

stfx = 1e6
stfy = 0.8e6
bearing0 = rs.BearingElement(0, kxx=stfx, kyy=stfy, cxx=0, n_link=7)
bearing1 = rs.BearingElement(6, kxx=stfx, kyy=stfy, cxx=0)
bearing2 = rs.BearingElement(7, kxx=stfx, kyy=stfy, cxx=0)

bearings = [bearing0, bearing1, bearing2]

pm0 = rs.PointMass(n=7, m=30)
pointmass = [pm0]

rotor1 = rs.Rotor(shaft_elem, disks, bearings, pointmass)

print("Rotor total mass = ", np.round(rotor1.m, 2))
print("Rotor center of gravity =", np.round(rotor1.CG, 2))

Rotor total mass =  88.18
Rotor center of gravity = 0.75


# Section 2: Visualizing the rotor model

It is interesting to plot the rotor to check if the geometry checks with what you wanted to the model. Use the `.plot_rotor()` method to create a plot.

`nodes` argument is useful when your model has lots of nodes and the visualization of nodes label may be confusing. Set an increment to the plot nodes label

ROSS uses **_Plotly_** as main plotting library, so you can hover the mouse icon over the shaft, disk and point mass elements to check some of their parameters.

In [3]:
rotor1.plot_rotor()

# Section 3: Adding new nodes

You can add new nodes to the model based on a list of their positions by using the method `.add_nodes()`:

In [4]:
node_position = [0.42, 1.1]
rotor1_new = rotor1.add_nodes(node_position)

rotor1_new.plot_rotor()

# Section 4: Adding new elements

You can add new elements (e.g.: `DiskElement`, `BearingElement`, `SealElement`, `PointMass`) to the model based on a list of elements by using the method `.add_elements()`.

It is valid for `ShaftElement` only when appending new shaft elements to the end of the shaft or inserting them around the existing shaft.

In [5]:
disk = rs.DiskElement(n=4, m=32, Id=0.2, Ip=0.3)
rotor1_new = rotor1_new.add_elements(
    [
        disk,
    ]
)

rotor1_new.plot_rotor()

# Section 5: Creating a rotor from sections

An alternative to build rotor models is dividing the rotor in sections. Each section gets the same number of shaft elements.

There's an important difference in this class method when placing disks and bearings. The argument `n` will refer, not to the element node, but to the section node. So if your model has 3 sections with 4 elements each, there're 4 section nodes and 13 element nodes.

Let's repeat the rotor model from the last example, but using `.from_section()` class method, without the support.

In [6]:
i_d = 0
o_d = 0.05

# inner diameter of each section
i_ds_data = [0, 0, 0]
# outer diameter of each section
o_ds_data = [0.05, 0.05, 0.05]
# length of each section
leng_data = [0.5, 0.5, 0.5]

material_data = [steel, steel, steel]
# material_data = steel
stfx = 1e6
stfy = 0.8e6

# n = 0 refers to the section 0, first node
bearing0 = rs.BearingElement(n=0, kxx=stfx, kyy=stfy, cxx=1e3)

# n = 3 refers to the section 2, last node
bearing1 = rs.BearingElement(n=3, kxx=stfx, kyy=stfy, cxx=1e3)
bearings = [bearing0, bearing1]

# n = 1 refers to the section 1, first node
disk0 = rs.DiskElement.from_geometry(
    n=1, material=steel, width=0.07, i_d=0.05, o_d=0.28
)

# n = 2 refers to the section 2, first node
disk1 = rs.DiskElement.from_geometry(
    n=2, material=steel, width=0.07, i_d=0.05, o_d=0.28
)
disks = [disk0, disk1]

rotor2 = rs.Rotor.from_section(
    brg_seal_data=bearings,
    disk_data=disks,
    idl_data=i_ds_data,
    leng_data=leng_data,
    odl_data=o_ds_data,
    nel_r=4,
    material_data=steel,
)

print("Rotor total mass = ", np.round(rotor2.m, 2))
print("Rotor center of gravity =", np.round(rotor2.CG, 2))

rotor2.plot_rotor()

Rotor total mass =  88.18
Rotor center of gravity = 0.75


# Section 6: Saving a rotor model

You can save a rotor model using the method `.save()`. This method saves the each element type and the rotor object in different *.toml* files. 

You just need to input a name and the diretory, where it will be saved. If you don't input a `file_path`, the rotor model is saved inside the **_ross_** folder.

To save the `rotor2` we can use:

In [7]:
rotor2.save("rotor2.toml")

# Section 7: Loading a rotor model

You can load a rotor model using the method `.load()`. This method loads a previously saved rotor model. So, you just need to input the file path to the method.

Now, let's load the `rotor2` we saved before:

In [8]:
rotor2_1 = rs.Rotor.load("rotor2.toml")
rotor2_1 == rotor2

True

# Section 8: Concatenating rotors

You can concatenate a sequence of rotors into a single rotor by using the `.concatenate()` class method or by summing the rotor objects. Concatenation joins shaft lines end to end. For shafts that run in parallel and are coupled by gears, use `MultiRotor` instead — see [Tutorial - MultiRotor System](tutorial_multirotor.ipynb).

In [9]:
rotor3 = rs.Rotor.concatenate(rotor1, rotor2)
rotor3.plot_rotor()

In [10]:
rotor3 = rotor1 + rotor2
rotor3.plot_rotor()

To model an induction motor, with a lumped or a finite element mechanical model, see [Tutorial - Three-Phase Induction Motor](tutorial_motor.ipynb).